In [ ]:
# Imports
import spacy
from spacy.training.example import Example
from spacy.util import minibatch, compounding
import random
import pandas as pd

In [ ]:
# Model
# Base model
nlp = spacy.blank('es')

# NER component
ner = nlp.add_pipe('ner')

train_data = pd.read_csv("data.csv")

# Add labels to the model
for _, annotations in train_data:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])

# Optimizer
optimizer = nlp.begin_training()


In [ ]:
# Training
n_iter = 100
for i in range(n_iter):
    random.shuffle(train_data)
    losses = {}
    batches = minibatch(train_data, size=compounding(4.0, 32.0, 1.001))
    for batch in batches:
        for text, annotations in batch:
            doc = nlp.make_doc(text)
            example = Example.from_dict(doc, annotations)
            nlp.update([example], drop=0.5, losses=losses)
    print(f"Iteración {i + 1}, Pérdida: {losses['ner']}")

# Save the model
nlp.to_disk("./modelo_ner")
print("Modelo guardado en ./modelo_ner")

In [ ]:
# Test
doc = nlp("Google ha lanzado un nuevo producto")
for ent in doc.ents:
    print(ent.text, ent.label_)